# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

### Dataset Source
The dataset metadata and structure are defined via a [Croissant schema](https://mlcommons.org/croissant/) available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview

List available record sets, fields, and their `@id` values.

In [ ]:
# List all record sets in the dataset
print("# Available record sets:")
record_sets_info = {}
for recset in metadata.record_sets:
    print(f"Record Set name: {recset.name} | @id: {recset.id}")
    # For each record set, list its fields by id and name
    record_sets_info[recset.id] = [field.id for field in recset.fields]
    print("  Fields:")
    for field in recset.fields:
        print(f"    - {field.name} (@id: {field.id})")

## 3. Data Extraction

We'll extract data from one or more record sets. Please refer to the `@id` values from above when specifying the record set and fields.

In [ ]:
# Decide which record set(s) to load. For the FAIR^2 dataset, there may be one to several.
record_set_ids = [recset.id for recset in metadata.record_sets]  # List of all ids

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  DataFrame with {df.shape[0]} rows and columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print("  No records found.")

## 4. Exploratory Data Analysis (EDA)

We'll inspect the first available non-empty record set. Let's apply numeric transformations and group by a selected field.

> **Note:** Replace the placeholders with valid field `@id`s from the output above for deeper exploration.

In [ ]:
# Identify a numeric field (e.g., coverage_percent, mw, number_peptides, etc.)
# For illustration, let's choose the first available numeric column.

target_record_set_id = None
for record_set_id, df in dataframes.items():
    numeric_cols = df.select_dtypes(include=['number']).columns
    if len(numeric_cols) > 0:
        target_record_set_id = record_set_id
        numeric_field_id = numeric_cols[0]  # Pick the first numeric field found
        break
if target_record_set_id is None:
    print("No numeric fields found in the available record sets.")
else:
    print(f"Analyzing record set @id: {target_record_set_id}")
    print(f"Numeric field selected: {numeric_field_id}")

    # Filter records where the numeric field is above a threshold
    threshold = dataframes[target_record_set_id][numeric_field_id].mean()
    filtered_df = dataframes[target_record_set_id][dataframes[target_record_set_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt to group by the first string/categorical column
    group_field_candidates = filtered_df.select_dtypes(include=['object', 'category']).columns
    if len(group_field_candidates) > 0:
        group_field = group_field_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean {numeric_field_id} by {group_field}:")
        display(grouped_df.head())
    else:
        print("No suitable string/categorical field to group by.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field. We'll use matplotlib to plot the histogram for the normalized field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if target_record_set_id is not None and f"{numeric_field_id}_normalized" in filtered_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of normalized {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id} (normalized)")
    plt.ylabel("Frequency")
    plt.show()
else:
    print("No normalized numeric field to plot.")

## 6. Conclusion

In this notebook, we loaded and explored the FAIR^2 dataset using the `mlcroissant` library:

- The dataset provides mass spectrometry results of human mast cell extracellular vesicles.
- We programmatically listed the available record sets and fields using their `@id`s for precise referencing.
- A numeric field was selected for detailed analysis, including filtering, normalization, grouping, and visualization.

The FAIR^2 format and Croissant schema provide robust ways to describe, find, and process open scientific datasets.

> **Next steps:** You can further explore relationships within the data, join across record sets using `@id`s, or build predictive models using the processed data. Review the [FAIR^2 metadata](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) and Croissant schema for additional derived fields, annotations, and provenance.